In [ ]:
"""Day 17 / Session 1 - Lab Assignment 2: Backprop by Hand - a 2-layer MLP that learns XOR."""
import numpy as np


def sigmoid(z):
    z = np.asarray(z, dtype=float)
    out = np.empty_like(z)
    pos = z >= 0
    out[pos] = 1.0 / (1.0 + np.exp(-z[pos]))
    ez = np.exp(z[~pos])
    out[~pos] = ez / (1.0 + ez)
    return out


def forward(X, params):
    z1 = X @ params["W1"] + params["b1"]
    a1 = np.tanh(z1)
    z2 = a1 @ params["W2"] + params["b2"]
    a2 = sigmoid(z2)
    return {"z1": z1, "a1": a1, "z2": z2, "a2": a2}


def bce_loss(y, y_hat, eps=1e-12):
    p = np.clip(y_hat, eps, 1.0 - eps)
    return float(-np.mean(y * np.log(p) + (1.0 - y) * np.log(1.0 - p)))


def backprop(X, y, params, cache):
    m = X.shape[0]
    dz2 = (cache["a2"] - y) / m                          # sigmoid output + BCE loss
    dW2 = cache["a1"].T @ dz2
    db2 = np.sum(dz2, axis=0, keepdims=True)
    da1 = dz2 @ params["W2"].T
    dz1 = da1 * (1.0 - cache["a1"] ** 2)                 # tanh derivative
    dW1 = X.T @ dz1
    db1 = np.sum(dz1, axis=0, keepdims=True)
    return {"dW1": dW1, "db1": db1, "dW2": dW2, "db2": db2}


def gradient_step(params, grads, lr):
    return {name: params[name] - lr * grads["d" + name] for name in params}